In [1]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AI_Text_Detector"
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")

os.makedirs(DATASET_DIR, exist_ok=True)

print("Dataset directory:", DATASET_DIR)

Dataset directory: /content/drive/MyDrive/AI_Text_Detector\dataset


In [2]:
from google.colab import drive

drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/AI_Text_Detector"

print("Project directory:")
print(PROJECT_DIR)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os

folders = [
    "dataset",
    "model",
    "notebooks",
    "backend",
    "frontend"
]

for folder in folders:
    path = os.path.join(PROJECT_DIR, folder)
    print(f"{folder}: {'OK' if os.path.exists(path) else 'NOT FOUND'}")

dataset: OK
model: OK
notebooks: OK
backend: OK
frontend: OK


In [ ]:
!pip -q install datasets pandas scikit-learn

In [ ]:
import pandas as pd
import numpy as np

from datasets import load_dataset
from sklearn.model_selection import train_test_split

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [ ]:
from datasets import load_dataset

print("Loading HC3 dataset...")

hc3 = load_dataset(
    "Hello-SimpleAI/HC3",
    revision="refs/convert/parquet",
    split="train"
)

print(hc3)

Loading HC3 dataset...


all/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 39.3MB            

all/train/0000.parquet: downloading bytes:           |  0.00B            

finance/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 5.18MB            

finance/train/0000.parquet: downloading bytes:           |  0.00B            

...

reddit_eli5/train/0000.parquet: downloading bytes:           |  0.00B            

wiki_csai/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.08MB            

wiki_csai/train/0000.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source'],
    num_rows: 48644
})


In [ ]:
print(hc3)

Dataset({
    features: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source'],
    num_rows: 48644
})


In [ ]:
print("===== HC3 SAMPLE CHECK =====")

print("\n--- HUMAN ANSWER ---")
print(hc3[0]["human_answers"])

print("\n--- CHATGPT ANSWER ---")
print(hc3[0]["chatgpt_answers"])

===== HC3 SAMPLE CHECK =====

--- HUMAN ANSWER ---
['Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence , # 1 best seller .', "If you 're hearing about it , it 's because it was a very good or very well - publicized book ( or both ) , and almost every good or well - publicized book will be # 1 on the NY Times bestseller list for at least a little bit . Kindof like how almost every big or good movies are # 1 at the box office on their opening weekend .", "One reaso

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("===== CREATING FINAL 10K DATASET =====")

# -----------------------------
# Create Human texts
# -----------------------------
human_texts = []

for answers in hc3["human_answers"]:
    if answers:
        for text in answers:
            if text and str(text).strip():
                human_texts.append(str(text).strip())


# -----------------------------
# Create AI texts
# -----------------------------
ai_texts = []

for answers in hc3["chatgpt_answers"]:
    if answers:
        for text in answers:
            if text and str(text).strip():
                ai_texts.append(str(text).strip())


print("Available Human texts:", len(human_texts))
print("Available AI texts:", len(ai_texts))


# -----------------------------
# Select 5000 + 5000
# -----------------------------
human_df = pd.DataFrame({
    "text": human_texts[:5000],
    "label": 0
})

ai_df = pd.DataFrame({
    "text": ai_texts[:5000],
    "label": 1
})


# -----------------------------
# Combine and shuffle
# -----------------------------
final_df = pd.concat(
    [human_df, ai_df],
    ignore_index=True
)

final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)


print("Total final samples:", len(final_df))
print("\nLabel distribution:")
print(final_df["label"].value_counts())


# -----------------------------
# Train / Validation / Test
# 80 / 10 / 10
# -----------------------------
train_df, temp_df = train_test_split(
    final_df,
    test_size=0.20,
    random_state=42,
    stratify=final_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)


print("\n===== DATASET SPLIT =====")
print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())

===== CREATING FINAL 10K DATASET =====
Available Human texts: 117092
Available AI texts: 53770
Total final samples: 10000

Label distribution:
label
1    5000
0    5000
Name: count, dtype: int64

===== DATASET SPLIT =====
Training: 8000
Validation: 1000
Testing: 1000

Training labels:
label
0    4000
1    4000
Name: count, dtype: int64

Validation labels:
label
1    500
0    500
Name: count, dtype: int64

Testing labels:
label
1    500
0    500
Name: count, dtype: int64


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/AI_Text_Detector"
DATASET_DIR = os.path.join(PROJECT_DIR, "dataset")

os.makedirs(DATASET_DIR, exist_ok=True)

train_df.to_csv(
    os.path.join(DATASET_DIR, "train.csv"),
    index=False
)

val_df.to_csv(
    os.path.join(DATASET_DIR, "validation.csv"),
    index=False
)

test_df.to_csv(
    os.path.join(DATASET_DIR, "test.csv"),
    index=False
)

print("===== DATASET SAVED TO GOOGLE DRIVE =====")
print("train.csv      :", len(train_df))
print("validation.csv :", len(val_df))
print("test.csv       :", len(test_df))
print("Location       :", DATASET_DIR)

===== DATASET SAVED TO GOOGLE DRIVE =====
train.csv      : 8000
validation.csv : 1000
test.csv       : 1000
Location       : /content/drive/MyDrive/AI_Text_Detector/dataset


In [ ]:
import os

print("===== CHECKING DATASET IN GOOGLE DRIVE =====")

for file in os.listdir(DATASET_DIR):
    path = os.path.join(DATASET_DIR, file)
    print(file, "->", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

===== CHECKING DATASET IN GOOGLE DRIVE =====
processed_dataset.csv -> 8.26 MB
train.csv -> 6.21 MB
validation.csv -> 0.77 MB
test.csv -> 0.77 MB
final_dataset -> 0.0 MB


In [ ]:
print("===== ANSWER COUNT CHECK =====")

human_count = sum(
    len(x) for x in hc3["human_answers"]
)

ai_count = sum(
    len(x) for x in hc3["chatgpt_answers"]
)

print("Total Human answers:", human_count)
print("Total AI answers:", ai_count)

===== ANSWER COUNT CHECK =====
Total Human answers: 117092
Total AI answers: 53806


In [ ]:
import pandas as pd
import random

print("===== CREATING BALANCED DATASET =====")

human_texts = []
ai_texts = []

# Collect Human answers
for answers in hc3["human_answers"]:
    for answer in answers:
        if isinstance(answer, str) and answer.strip():
            human_texts.append(answer.strip())

# Collect AI answers
for answers in hc3["chatgpt_answers"]:
    for answer in answers:
        if isinstance(answer, str) and answer.strip():
            ai_texts.append(answer.strip())

print("Usable Human texts:", len(human_texts))
print("Usable AI texts:", len(ai_texts))

# Shuffle
random.seed(42)

random.shuffle(human_texts)
random.shuffle(ai_texts)

# Select 5,000 each
human_sample = human_texts[:5000]
ai_sample = ai_texts[:5000]

# Create DataFrames
human_df = pd.DataFrame({
    "text": human_sample,
    "label": 0
})

ai_df = pd.DataFrame({
    "text": ai_sample,
    "label": 1
})

# Combine
final_df = pd.concat(
    [human_df, ai_df],
    ignore_index=True
)

# Shuffle final dataset
final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\n===== FINAL DATASET =====")
print("Total samples:", len(final_df))
print("\nLabel distribution:")
print(final_df["label"].value_counts())

print("\nSample:")
print(final_df.head())

===== CREATING BALANCED DATASET =====
Usable Human texts: 117092
Usable AI texts: 53770

===== FINAL DATASET =====
Total samples: 10000

Label distribution:
label
1    5000
0    5000
Name: count, dtype: int64

Sample:
                                                text  label
0  Railroad tracks are usually built on top of a ...      1
1  I 've only taken some basic psychology classes...      0
2  Water can absorb a lot of the heat given by fi...      0
3  There are some good points in this thread , bu...      0
4  One is "US Government Standard Bathroom Malodo...      0


In [ ]:
print("===== EMPTY TEXT CHECK =====")

print("Empty training texts:",
      train_df["text"].isna().sum())

print("Empty validation texts:",
      val_df["text"].isna().sum())

print("Empty testing texts:",
      test_df["text"].isna().sum())

===== EMPTY TEXT CHECK =====
Empty training texts: 0
Empty validation texts: 0
Empty testing texts: 0


In [ ]:
print("===== TEXT LENGTH CHECK =====")

train_lengths = train_df["text"].str.split().str.len()

print("Minimum words:", train_lengths.min())
print("Maximum words:", train_lengths.max())
print("Average words:", round(train_lengths.mean(), 2))

===== TEXT LENGTH CHECK =====
Minimum words: 8
Maximum words: 2171
Average words: 148.6


In [ ]:
print("===== DATASET VERIFICATION =====")

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())

print("\nColumns:")
print(train_df.columns.tolist())

===== DATASET VERIFICATION =====

Training labels:
label
0    4000
1    4000
Name: count, dtype: int64

Validation labels:
label
1    500
0    500
Name: count, dtype: int64

Testing labels:
label
1    500
0    500
Name: count, dtype: int64

Columns:
['text', 'label']


In [ ]:
from datasets import Dataset

print("===== CONVERTING TO HUGGINGFACE DATASETS =====")

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print("Train dataset:")
print(train_dataset)

print("\nValidation dataset:")
print(val_dataset)

print("\nTest dataset:")
print(test_dataset)

===== CONVERTING TO HUGGINGFACE DATASETS =====
Train dataset:
Dataset({
    features: ['text', 'label'],
    num_rows: 8000
})

Validation dataset:
Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})

Test dataset:
Dataset({
    features: ['text', 'label'],
    num_rows: 1000
})


In [ ]:
from transformers import BertTokenizer

print("===== LOADING BERT TOKENIZER =====")

tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("Tokenizer loaded successfully.")

===== LOADING BERT TOKENIZER =====
Tokenizer loaded successfully.


In [ ]:
print("===== TOKENIZING DATASET =====")

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

print("\nTokenization completed successfully.")

print("\nTrain columns:")
print(train_tokenized.column_names)

===== TOKENIZING DATASET =====


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]


Tokenization completed successfully.

Train columns:
['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
from transformers import BertForSequenceClassification

print("===== LOADING BERT MODEL =====")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# IMPORTANT: Keep label mapping consistent
model.config.id2label = {
    0: "HUMAN",
    1: "AI_GENERATED"
}

model.config.label2id = {
    "HUMAN": 0,
    "AI_GENERATED": 1
}

print("BERT model loaded successfully.")
print("Labels:", model.config.id2label)

===== LOADING BERT MODEL =====


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded successfully.
Labels: {0: 'HUMAN', 1: 'AI_GENERATED'}


In [ ]:
print("===== PREPARING DATA FORMAT =====")

train_tokenized = train_tokenized.rename_column(
    "label",
    "labels"
)

val_tokenized = val_tokenized.rename_column(
    "label",
    "labels"
)

test_tokenized = test_tokenized.rename_column(
    "label",
    "labels"
)

train_tokenized.set_format("torch")
val_tokenized.set_format("torch")
test_tokenized.set_format("torch")

print("Dataset format prepared successfully.")

===== PREPARING DATA FORMAT =====
Dataset format prepared successfully.


In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    accuracy = accuracy_score(
        labels,
        predictions
    )

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("Evaluation metrics configured successfully.")

Evaluation metrics configured successfully.


In [ ]:
from transformers import TrainingArguments

print("===== CONFIGURING TRAINING =====")

training_args = TrainingArguments(
    output_dir="/content/bert_ai_detector",

    num_train_epochs=3,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    learning_rate=2e-5,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    logging_strategy="epoch",

    report_to="none"
)

print("Training configuration ready.")

===== CONFIGURING TRAINING =====
Training configuration ready.


In [ ]:
from transformers import Trainer

print("===== CREATING TRAINER =====")

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    compute_metrics=compute_metrics
)

print("Trainer created successfully.")

===== CREATING TRAINER =====
Trainer created successfully.


In [ ]:
from transformers import Trainer

print("===== CREATING TRAINER =====")

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    compute_metrics=compute_metrics
)

print("Trainer created successfully.")

===== CREATING TRAINER =====
Trainer created successfully.


In [ ]:
import torch
import torchvision

print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
Torchvision version: 0.26.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -U "transformers>=5.0.0"

In [ ]:
import datasets

# Disable torchvision integration for this text-only BERT project
datasets.config.TORCHVISION_AVAILABLE = False

print("Torchvision integration disabled.")
print("Ready for BERT text training.")

Torchvision integration disabled.
Ready for BERT text training.


In [ ]:
print("===== STARTING BERT TRAINING =====")

train_result = trainer.train()

print("\n===== TRAINING COMPLETED =====")
print(train_result)

===== STARTING BERT TRAINING =====


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.060663,0.130542,0.976000,0.954198,1.000000,0.976562
2,0.005632,0.082619,0.988000,0.976562,1.000000,0.988142
3,0.001563,0.023340,0.995000,0.990099,1.000000,0.995025


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.060663,0.130542,0.976000,0.954198,1.000000,0.976562
2,0.005632,0.082619,0.988000,0.976562,1.000000,0.988142
3,0.001563,0.023340,0.995000,0.990099,1.000000,0.995025



===== TRAINING COMPLETED =====
TrainOutput(global_step=1500, training_loss=0.022619465748469035, metrics={'train_runtime': 1111.0709, 'train_samples_per_second': 21.601, 'train_steps_per_second': 1.35, 'total_flos': 3157332664320000.0, 'train_loss': 0.022619465748469035, 'epoch': 3.0})


In [ ]:
import os

MODEL_PATH = "/content/drive/MyDrive/AI_Text_Detector/model/bert_ai_detector"

os.makedirs(MODEL_PATH, exist_ok=True)

print("===== SAVING TRAINED MODEL =====")

trainer.save_model(MODEL_PATH)
tokenizer.save_pretrained(MODEL_PATH)

print("===== MODEL SAVED SUCCESSFULLY =====")
print("Model path:", MODEL_PATH)

===== SAVING TRAINED MODEL =====


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

===== MODEL SAVED SUCCESSFULLY =====
Model path: /content/bert_ai_detector


In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

MODEL_PATH = "/content/bert_ai_detector"

print("===== LOADING SAVED MODEL =====")

tokenizer_test = BertTokenizer.from_pretrained(MODEL_PATH)
model_test = BertForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_test.to(device)
model_test.eval()

print("Model loaded successfully.")
print("Device:", device)


def test_prediction(text):

    inputs = tokenizer_test(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model_test(**inputs)
        probabilities = torch.softmax(
            outputs.logits,
            dim=-1
        )

    predicted_class = torch.argmax(
        probabilities,
        dim=-1
    ).item()

    confidence = probabilities[0, predicted_class].item() * 100

    label = model_test.config.id2label.get(
        predicted_class,
        str(predicted_class)
    )

    return label, confidence


print("\n===== TEST 1 =====")

text1 = """
Artificial intelligence is transforming modern society by enabling
organizations to automate complex tasks, improve decision-making,
and enhance productivity across various industries.
"""

label, confidence = test_prediction(text1)

print("Prediction:", label)
print("Confidence:", round(confidence, 2), "%")

===== LOADING SAVED MODEL =====


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully.
Device: cuda

===== TEST 1 =====
Prediction: HUMAN
Confidence: 99.97 %


In [ ]:
print("===== MULTIPLE TEXT TEST =====")

test_texts = [

    # Test 1
    """
    Artificial intelligence is transforming modern society by enabling
    organizations to automate complex tasks, improve decision-making,
    and enhance productivity across various industries.
    """,

    # Test 2
    """
    Machine learning models can analyze large amounts of data and
    identify patterns that help organizations make better decisions.
    """,

    # Test 3
    """
    Yesterday I went to college and attended my computer networks class.
    After the class, I had lunch with my friends and came back home.
    """,

    # Test 4
    """
    I really enjoyed the cricket match yesterday because the players
    performed extremely well and the final over was very exciting.
    """
]

for i, text in enumerate(test_texts, 1):

    label, confidence = test_prediction(text)

    print("\n-----------------------------")
    print("TEST", i)
    print("Text:", text.strip()[:100], "...")
    print("Prediction:", label)
    print("Confidence:", round(confidence, 2), "%")

===== MULTIPLE TEXT TEST =====

-----------------------------
TEST 1
Text: Artificial intelligence is transforming modern society by enabling
    organizations to automate com ...
Prediction: HUMAN
Confidence: 99.97 %

-----------------------------
TEST 2
Text: Machine learning models can analyze large amounts of data and
    identify patterns that help organi ...
Prediction: HUMAN
Confidence: 99.31 %

-----------------------------
TEST 3
Text: Yesterday I went to college and attended my computer networks class.
    After the class, I had lunc ...
Prediction: HUMAN
Confidence: 99.98 %

-----------------------------
TEST 4
Text: I really enjoyed the cricket match yesterday because the players
    performed extremely well and th ...
Prediction: HUMAN
Confidence: 99.98 %


In [ ]:
print("===== TEST DATASET PREDICTION CHECK =====")

from collections import Counter
import torch

predictions = []
actual_labels = []

for text, label in zip(test_df["text"], test_df["label"]):

    inputs = tokenizer_test(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model_test(**inputs)

    predicted_class = torch.argmax(
        outputs.logits,
        dim=-1
    ).item()

    predictions.append(predicted_class)
    actual_labels.append(int(label))


print("Actual label distribution:")
print(Counter(actual_labels))

print("\nPredicted label distribution:")
print(Counter(predictions))

correct = sum(
    p == a
    for p, a in zip(predictions, actual_labels)
)

accuracy = correct / len(actual_labels) * 100

print("\nCorrect predictions:", correct)
print("Total test samples:", len(actual_labels))
print("Test Accuracy:", round(accuracy, 2), "%")

===== TEST DATASET PREDICTION CHECK =====
Actual label distribution:
Counter({1: 500, 0: 500})

Predicted label distribution:
Counter({1: 507, 0: 493})

Correct predictions: 993
Total test samples: 1000
Test Accuracy: 99.3 %


In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

print("===== MODEL EVALUATION =====")

print("\nClassification Report:\n")

print(
    classification_report(
        actual_labels,
        predictions,
        target_names=["HUMAN", "AI_GENERATED"],
        digits=4
    )
)

print("\nConfusion Matrix:\n")

cm = confusion_matrix(
    actual_labels,
    predictions
)

print(cm)

===== MODEL EVALUATION =====

Classification Report:

              precision    recall  f1-score   support

       HUMAN     1.0000    0.9860    0.9930       500
AI_GENERATED     0.9862    1.0000    0.9930       500

    accuracy                         0.9930      1000
   macro avg     0.9931    0.9930    0.9930      1000
weighted avg     0.9931    0.9930    0.9930      1000


Confusion Matrix:

[[493   7]
 [  0 500]]


In [ ]:
import shutil
import os

MODEL_PATH = "/content/bert_ai_detector"
ZIP_PATH = "/content/bert_ai_detector"

print("===== CREATING MODEL ZIP =====")

shutil.make_archive(
    ZIP_PATH,
    "zip",
    MODEL_PATH
)

print("===== MODEL ZIP CREATED =====")
print("File:", ZIP_PATH + ".zip")
print("Size:", round(os.path.getsize(ZIP_PATH + ".zip") / (1024 * 1024), 2), "MB")

===== CREATING MODEL ZIP =====


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/bert_ai_detector'

===== PREPARING MODEL DOWNLOAD =====


FileNotFoundError: Cannot find file: /content/mydrive/bert_ai_detector.zip